<a href="https://colab.research.google.com/github/sr606/LLM/blob/main/mermaid_v7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
🔵 1️⃣ main.py
# main.py
from agent import run_agent

if __name__ == "__main__":
    run_agent()
🔵 2️⃣ agent.py (FULL CLEAN VERSION)
import os
import json

from parser import split_into_stages
from deterministic_parser import parse_stage_technical
from llm_service import LLMService

from graph_builder import build_graph
from pipeline_detector import detect_pipelines
from pipeline_name import generate_pipeline_name

from renderer import render_detailed_pipeline
from high_level_renderer import render_high_level_architecture
from technical_pdf_writer import generate_technical_pdf


INPUT_FOLDER = "../data/input"
OUTPUT_FOLDER = "../data/output"


def run_agent():
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)

    llm = LLMService()

    for file in os.listdir(INPUT_FOLDER):
        if not file.endswith(".txt"):
            continue

        input_path = os.path.join(INPUT_FOLDER, file)

        with open(input_path, "r", encoding="utf-8") as f:
            pseudocode = f.read()

        print(f"\nProcessing: {file}")

        # ---------------------------
        # Split stages
        # ---------------------------
        stages = split_into_stages(pseudocode)

        stage_metadata = []
        technical_metadata = []

        # ---------------------------
        # Process each stage
        # ---------------------------
        for stage in stages:
            stage_id = stage["stage_id"]
            block = stage["block"]

            # Deterministic technical parsing
            tech = parse_stage_technical(stage_id, block)
            technical_metadata.append(tech)

            # LLM only for summary
            summary = llm.generate_summary(block)

            stage_metadata.append({
                "stage_name": stage_id,
                "stage_type": tech["stage_type"],
                "summary": summary
            })

        # ---------------------------
        # Save metadata
        # ---------------------------
        base_name = file.replace(".txt", "")
        metadata_path = os.path.join(OUTPUT_FOLDER, f"{base_name}_metadata.json")

        with open(metadata_path, "w", encoding="utf-8") as f:
            json.dump(technical_metadata, f, indent=4)

        # ---------------------------
        # Build stage graph
        # ---------------------------
        stage_id_order = [s["stage_id"] for s in stages]
        graph = build_graph(pseudocode, stage_metadata, stage_id_order)

        pipelines = detect_pipelines(graph, include_singletons=False)

        pipeline_names = [
            generate_pipeline_name(graph, p)
            for p in pipelines
        ]

        # ---------------------------
        # Render Detailed Stage Diagram
        # ---------------------------
        render_detailed_pipeline(
            graph,
            pipelines,
            pipeline_names,
            os.path.join(OUTPUT_FOLDER, f"{base_name}_detailed")
        )

        # ---------------------------
        # Render High Level Architecture
        # ---------------------------
        render_high_level_architecture(
            technical_metadata,
            os.path.join(OUTPUT_FOLDER, f"{base_name}_high_level")
        )

        # ---------------------------
        # Generate Technical PDF
        # ---------------------------
        generate_technical_pdf(
            technical_metadata,
            os.path.join(OUTPUT_FOLDER, f"{base_name}_technical.pdf")
        )

        print(f"Completed: {file}")
🔵 3️⃣ deterministic_parser.py
import re


def parse_stage_technical(stage_id: str, stage_block: str):
    result = {
        "stage_id": stage_id,
        "stage_type": extract_stage_type(stage_block),
        "input_datasets": [],
        "output_datasets": [],
        "joins": [],
        "filters": [],
        "aggregations": [],
        "column_mappings": []
    }

    # Inputs
    result["input_datasets"] = re.findall(
        r"Input:\s*←\s*dataset_\d+\s*\(([^)]+)\)",
        stage_block
    )

    # Outputs
    result["output_datasets"] = re.findall(
        r"Output:\s*→\s*dataset_\d+\s*\(([^)]+)\)",
        stage_block
    )

    # Joins
    join_pattern = r"(INNER|LEFT|RIGHT|FULL|LEFT OUTER|RIGHT OUTER)?\s*JOIN\s+([\w\.]+)\s+ON\s+(.*?)(?:\n|WHERE|GROUP BY|ORDER BY)"
    joins = re.findall(join_pattern, stage_block, re.IGNORECASE | re.DOTALL)

    for jt, table, condition in joins:
        result["joins"].append({
            "join_type": (jt or "JOIN").upper(),
            "table": table.strip(),
            "condition": condition.strip()
        })

    # Filters
    where_pattern = r"WHERE\s+(.*?)(?:GROUP BY|ORDER BY|$)"
    filters = re.findall(where_pattern, stage_block, re.IGNORECASE | re.DOTALL)
    result["filters"] = [f.strip() for f in filters]

    # Aggregations
    agg_pattern = r"(SUM|COUNT|AVG|MIN|MAX)\((.*?)\)"
    aggs = re.findall(agg_pattern, stage_block, re.IGNORECASE)
    result["aggregations"] = [f"{f.upper()}({c})" for f, c in aggs]

    # Column mappings (Transformer stages)
    mapping_pattern = r"(\w+)\s*=\s*([A-Za-z0-9_\.]+)"
    mappings = re.findall(mapping_pattern, stage_block)

    for target, source in mappings:
        result["column_mappings"].append({
            "source": source,
            "target": target
        })

    return result


def extract_stage_type(block):
    match = re.search(r"StageType:\s*(\w+)", block)
    return match.group(1) if match else "Unknown"
🔵 4️⃣ llm_service.py (SUMMARY ONLY)
import os
from dotenv import load_dotenv
from openai import AzureOpenAI

load_dotenv()


class LLMService:

    def __init__(self):
        self.client = AzureOpenAI(
            api_key=os.getenv("AZURE_OPENAI_API_KEY"),
            api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
            azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        )
        self.deployment = os.getenv("AZURE_OPENAI_DEPLOYMENT")

    def generate_summary(self, stage_block: str):

        prompt = """
Summarize the following ETL stage in 3 concise bullet points.
Do not infer anything not present.
Be technical and precise.
Return only bullet points.
"""

        response = self.client.chat.completions.create(
            model=self.deployment,
            temperature=0.0,
            messages=[
                {"role": "system", "content": "You summarize ETL stages."},
                {"role": "user", "content": prompt + stage_block}
            ],
        )

        text = response.choices[0].message.content.strip()
        return [line.strip("- ").strip() for line in text.split("\n") if line.strip()]
🔵 5️⃣ renderer.py (Detailed Stage Diagram)
from graphviz import Digraph


def render_detailed_pipeline(graph, pipelines, pipeline_names, output_path):
    dot = Digraph("Detailed_Pipeline", format="pdf")
    dot.attr(rankdir="LR")

    dot.node_attr.update(
        shape="box",
        style="rounded,filled",
        fillcolor="#ECEFF1",
        fontname="Helvetica",
        fontsize="10"
    )

    for idx, pipeline in enumerate(pipelines):
        with dot.subgraph(name=f"cluster_{idx}") as sub:
            sub.attr(label=pipeline_names[idx], style="rounded")

            for stage_id in pipeline:
                node = graph.nodes[stage_id]
                label = f"{node.display_name}\n({node.stage_type})"
                sub.node(stage_id, label)

    for src, tgt in graph.edges:
        dot.edge(src, tgt)

    dot.render(output_path, cleanup=True)
🔵 6️⃣ high_level_renderer.py
from graphviz import Digraph


def render_high_level_architecture(technical_metadata, output_path):
    dot = Digraph("High_Level", format="pdf")
    dot.attr(rankdir="LR")

    dot.node("Sources", shape="box", style="filled", fillcolor="#BBDEFB")
    dot.node("Transformations", shape="box", style="filled", fillcolor="#FFE0B2")
    dot.node("Targets", shape="box", style="filled", fillcolor="#C8E6C9")

    dot.edge("Sources", "Transformations")
    dot.edge("Transformations", "Targets")

    dot.render(output_path, cleanup=True)
🔵 7️⃣ technical_pdf_writer.py
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, ListFlowable, ListItem
from reportlab.lib.styles import ParagraphStyle
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet


def generate_technical_pdf(technical_metadata, output_path):

    doc = SimpleDocTemplate(output_path)
    elements = []

    styles = getSampleStyleSheet()
    normal = styles["Normal"]

    for stage in technical_metadata:

        elements.append(Paragraph(f"<b>Stage:</b> {stage['stage_id']}", styles["Heading3"]))
        elements.append(Spacer(1, 10))

        bullets = []

        if stage["input_datasets"]:
            bullets.append(f"Inputs: {', '.join(stage['input_datasets'])}")

        if stage["output_datasets"]:
            bullets.append(f"Outputs: {', '.join(stage['output_datasets'])}")

        for j in stage["joins"]:
            bullets.append(f"{j['join_type']} JOIN {j['table']} ON {j['condition']}")

        for f in stage["filters"]:
            bullets.append(f"Filter: {f}")

        for a in stage["aggregations"]:
            bullets.append(f"Aggregation: {a}")

        for m in stage["column_mappings"][:10]:
            bullets.append(f"{m['source']} → {m['target']}")

        flow = ListFlowable(
            [ListItem(Paragraph(b, normal)) for b in bullets],
            bulletType='bullet'
        )

        elements.append(flow)
        elements.append(Spacer(1, 20))

    doc.build(elements)

In [ ]:
🔵 main.py
from agent import run_agent

if __name__ == "__main__":
    run_agent()
🔵 agent.py
import os
from parser import split_into_stages
from deterministic_parser import parse_stage
from graph_builder import build_hybrid_graph
from renderer import render_hybrid_diagram
from high_level_renderer import render_high_level


INPUT_FOLDER = "data/input"
OUTPUT_FOLDER = "data/output"


def run_agent():

    os.makedirs(OUTPUT_FOLDER, exist_ok=True)

    for file in os.listdir(INPUT_FOLDER):
        if not file.endswith(".txt"):
            continue

        with open(os.path.join(INPUT_FOLDER, file), "r", encoding="utf-8") as f:
            pseudocode = f.read()

        print(f"Processing: {file}")

        stages = split_into_stages(pseudocode)

        stage_data = []
        for stage in stages:
            parsed = parse_stage(stage["stage_id"], stage["block"])
            stage_data.append(parsed)

        graph = build_hybrid_graph(stage_data)

        base = file.replace(".txt", "")

        render_hybrid_diagram(graph, os.path.join(OUTPUT_FOLDER, f"{base}_hybrid"))
        render_high_level(stage_data, os.path.join(OUTPUT_FOLDER, f"{base}_high_level"))

        print(f"Completed: {file}")
🔵 parser.py
import re

def split_into_stages(text):

    pattern = r"// --- \[(.*?)\] ---([\s\S]*?)(?=// --- \[|$)"
    matches = re.findall(pattern, text)

    stages = []

    for header, body in matches:
        parts = header.split(":")
        if len(parts) < 2:
            continue

        stage_id = parts[1].strip()
        block = header + "\n" + body.strip()

        stages.append({
            "stage_id": stage_id,
            "block": block
        })

    return stages
🔵 deterministic_parser.py
import re

def parse_stage(stage_id, block):

    result = {
        "stage_id": stage_id,
        "stage_type": extract_stage_type(block),
        "inputs": [],
        "outputs": [],
        "transform_columns": []
    }

    # Inputs
    result["inputs"] = re.findall(
        r"Input:\s*←\s*dataset_\d+\s*\(([^)]+)\)",
        block
    )

    # Outputs
    result["outputs"] = re.findall(
        r"Output:\s*→\s*dataset_\d+\s*\(([^)]+)\)",
        block
    )

    # Column transformations
    mapping_pattern = r"^\s*(\w+)\s*=\s*([A-Za-z0-9_\.]+)"
    lines = block.split("\n")

    for line in lines:
        match = re.match(mapping_pattern, line.strip())
        if match:
            result["transform_columns"].append({
                "target": match.group(1),
                "source": match.group(2)
            })

    return result


def extract_stage_type(block):
    match = re.search(r"StageType:\s*(\w+)", block)
    return match.group(1) if match else "Unknown"
🔵 graph_model.py
class Graph:

    def __init__(self):
        self.nodes = {}  # name -> type
        self.edges = []  # (source, target)

    def add_node(self, name, node_type):
        self.nodes[name] = node_type

    def add_edge(self, source, target):
        if source != target:
            self.edges.append((source, target))
🔵 graph_builder.py
from graph_model import Graph


def build_hybrid_graph(stage_data):

    graph = Graph()

    for stage in stage_data:

        stage_id = stage["stage_id"]

        graph.add_node(stage_id, "stage")

        # Input datasets
        for dataset in stage["inputs"]:
            graph.add_node(dataset, "dataset")
            graph.add_edge(dataset, stage_id)

        # Output datasets
        for dataset in stage["outputs"]:
            graph.add_node(dataset, "dataset")
            graph.add_edge(stage_id, dataset)

    return graph
🔵 renderer.py (Hybrid Detailed Diagram)
from graphviz import Digraph


def render_hybrid_diagram(graph, output_path):

    dot = Digraph("Hybrid", format="pdf")
    dot.attr(rankdir="LR")

    for name, node_type in graph.nodes.items():

        if node_type == "dataset":
            dot.node(name, name, shape="cylinder",
                     style="filled", fillcolor="#BBDEFB")

        else:
            dot.node(name, name, shape="box",
                     style="rounded,filled", fillcolor="#FFE0B2")

    for src, tgt in graph.edges:
        dot.edge(src, tgt)

    dot.render(output_path, cleanup=True)
🔵 high_level_renderer.py
from graphviz import Digraph


def render_high_level(stage_data, output_path):

    dot = Digraph("HighLevel", format="pdf")
    dot.attr(rankdir="LR")

    sources = []
    transforms = []
    targets = []

    for stage in stage_data:

        if not stage["inputs"]:
            sources.append(stage["stage_id"])
        elif not stage["outputs"]:
            targets.append(stage["stage_id"])
        else:
            transforms.append(stage["stage_id"])

    dot.node("Sources", "\n".join(sources) or "None",
             shape="box", style="filled", fillcolor="#BBDEFB")

    dot.node("Transformations", "\n".join(transforms) or "None",
             shape="box", style="filled", fillcolor="#FFE0B2")

    dot.node("Targets", "\n".join(targets) or "None",
             shape="box", style="filled", fillcolor="#C8E6C9")

    dot.edge("Sources", "Transformations")
    dot.edge("Transformations", "Targets")

    dot.render(output_path, cleanup=True)
🔵 requirements.txt
graphviz
🚀 HOW TO RUN
cd project
python -m venv venv
venv\Scripts\activate
pip install -r requirements.txt
python main.py
🔥 FINAL RESULT

You will get:

data/output/
    Sample_Job_2_hybrid.pdf
    Sample_Job_2_high_level.pdf
🎯 Summary
Feature	Status
LLM used	❌ No
Deterministic	✅ Yes
Hybrid lineage	✅ Yes
Dataset + Stage integration	✅ Yes
Clean enterprise diagram	✅ Yes

If you want next level:

Add column-level lineage edges

Extract real SQL source tables

Detect joins and show them

Layered swimlane architecture

Tell me your next upgrade 🚀

In [ ]:
1️⃣ deterministic_parser.py (UPGRADED)

Now extracts:

SQL tables (FROM + JOIN)

datasets

file outputs

stage type

transform columns

import re

def parse_stage(stage_id, block):

    result = {
        "stage_id": stage_id,
        "stage_type": extract_stage_type(block),
        "sql_tables": [],
        "input_datasets": [],
        "output_datasets": [],
        "output_files": [],
        "transform_columns": []
    }

    # ----------------------------
    # Extract SQL tables (FROM + JOIN)
    # ----------------------------
    sql_tables = re.findall(r"\bFROM\s+([\w\.]+)", block, re.IGNORECASE)
    join_tables = re.findall(r"\bJOIN\s+([\w\.]+)", block, re.IGNORECASE)

    result["sql_tables"] = list(set(sql_tables + join_tables))

    # ----------------------------
    # Dataset Inputs
    # ----------------------------
    result["input_datasets"] = re.findall(
        r"Input:\s*←\s*dataset_\d+\s*\(([^)]+)\)",
        block
    )

    # ----------------------------
    # Dataset Outputs
    # ----------------------------
    result["output_datasets"] = re.findall(
        r"Output:\s*→\s*dataset_\d+\s*\(([^)]+)\)",
        block
    )

    # ----------------------------
    # File outputs
    # ----------------------------
    file_outputs = re.findall(r"Link File .*?:\s*(.*)", block)
    result["output_files"] = file_outputs

    # ----------------------------
    # Transformation mappings
    # ----------------------------
    mapping_pattern = r"^\s*(\w+)\s*=\s*([A-Za-z0-9_\.]+)"
    for line in block.split("\n"):
        m = re.match(mapping_pattern, line.strip())
        if m:
            result["transform_columns"].append({
                "target": m.group(1),
                "source": m.group(2)
            })

    return result


def extract_stage_type(block):
    m = re.search(r"StageType:\s*(\w+)", block)
    return m.group(1) if m else "Unknown"
2️⃣ graph_model.py
class Graph:

    def __init__(self):
        self.nodes = {}  # name -> type
        self.edges = []

    def add_node(self, name, node_type):
        self.nodes[name] = node_type

    def add_edge(self, source, target):
        if source != target:
            self.edges.append((source, target))
3️⃣ graph_builder.py (Hybrid Enterprise Version)
from graph_model import Graph

def build_hybrid_graph(stage_data):

    graph = Graph()

    dataset_producers = {}

    for stage in stage_data:

        sid = stage["stage_id"]
        stype = stage["stage_type"]

        graph.add_node(sid, "stage")

        # SQL Tables → Stage
        for table in stage["sql_tables"]:
            graph.add_node(table, "sql_table")
            graph.add_edge(table, sid)

        # Dataset Inputs → Stage
        for ds in stage["input_datasets"]:
            graph.add_node(ds, "dataset")
            graph.add_edge(ds, sid)

        # Stage → Dataset Outputs
        for ds in stage["output_datasets"]:
            graph.add_node(ds, "dataset")
            graph.add_edge(sid, ds)
            dataset_producers[ds] = sid

        # Stage → Output File
        for f in stage["output_files"]:
            graph.add_node(f, "file")
            graph.add_edge(sid, f)

    return graph
4️⃣ renderer.py (PROPER LAYERED HYBRID)
from graphviz import Digraph

def render_hybrid_diagram(graph, output_path):

    dot = Digraph("HybridLineage", format="pdf")
    dot.attr(rankdir="LR", splines="spline")

    # Layer groups
    sql_nodes = []
    stage_nodes = []
    dataset_nodes = []
    file_nodes = []

    for name, ntype in graph.nodes.items():

        if ntype == "sql_table":
            sql_nodes.append(name)
        elif ntype == "dataset":
            dataset_nodes.append(name)
        elif ntype == "file":
            file_nodes.append(name)
        else:
            stage_nodes.append(name)

    # SQL tables layer
    with dot.subgraph() as s:
        s.attr(rank="same")
        for n in sql_nodes:
            dot.node(n, n, shape="cylinder",
                     style="filled", fillcolor="#BBDEFB")

    # Stage layer
    for n in stage_nodes:
        dot.node(n, n, shape="box",
                 style="rounded,filled",
                 fillcolor="#FFE0B2")

    # Dataset layer
    for n in dataset_nodes:
        dot.node(n, n, shape="cylinder",
                 style="filled",
                 fillcolor="#E1F5FE")

    # File layer
    for n in file_nodes:
        dot.node(n, n, shape="folder",
                 style="filled",
                 fillcolor="#C8E6C9")

    # Edges
    for src, tgt in graph.edges:
        dot.edge(src, tgt)

    dot.render(output_path, cleanup=True)
5️⃣ high_level_renderer.py (True Layer View)
from graphviz import Digraph

def render_high_level(stage_data, output_path):

    dot = Digraph("HighLevel", format="pdf")
    dot.attr(rankdir="LR")

    sources = []
    transforms = []
    targets = []

    for stage in stage_data:

        stype = stage["stage_type"]

        if stype == "OracleConnector":
            sources.append(stage["stage_id"])
        elif stype == "CSeqFileStage":
            targets.append(stage["stage_id"])
        else:
            transforms.append(stage["stage_id"])

    dot.node("Sources", "\n".join(sources),
             shape="box", style="filled", fillcolor="#BBDEFB")

    dot.node("Transformations", "\n".join(transforms),
             shape="box", style="filled", fillcolor="#FFE0B2")

    dot.node("Targets", "\n".join(targets),
             shape="box", style="filled", fillcolor="#C8E6C9")

    dot.edge("Sources", "Transformations")
    dot.edge("Transformations", "Targets")

    dot.render(output_path, cleanup=True)
🔥 What You Now Have

Proper hybrid lineage:

SQL tables visible

Dataset nodes visible

Stage nodes visible

File outputs visible

Deterministic

Clean layering

Enterprise-grade

In [ ]:
📁 FINAL PROJECT STRUCTURE
project/
│
├── main.py
├── agent.py
│
├── parser.py
├── extractor.py
├── llm_normalizer.py
├── validator.py
│
├── graph_builder.py
├── renderer_unified.py
├── renderer_drilldown.py
│
├── requirements.txt
│
├── data/
│   ├── input/
│   └── output/
│
└── .env
1️⃣ main.py
from agent import run_agent

if __name__ == "__main__":
    run_agent()
2️⃣ agent.py
import os
from parser import split_into_stages
from extractor import extract_raw_blocks
from llm_normalizer import normalize_stage
from validator import validate_stage
from graph_builder import build_lineage_model
from renderer_unified import render_unified
from renderer_drilldown import render_drilldown

INPUT_FOLDER = "data/input"
OUTPUT_FOLDER = "data/output"


def run_agent():
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)

    for file in os.listdir(INPUT_FOLDER):
        if not file.endswith(".txt"):
            continue

        with open(os.path.join(INPUT_FOLDER, file), "r", encoding="utf-8") as f:
            pseudo = f.read()

        print(f"Processing {file}")

        stages = split_into_stages(pseudo)

        normalized_stages = []

        for stage in stages:
            raw = extract_raw_blocks(stage["block"])
            normalized = normalize_stage(stage["stage_id"], raw)
            validated = validate_stage(normalized, raw)
            normalized_stages.append(validated)

        lineage_model = build_lineage_model(normalized_stages)

        base = file.replace(".txt", "")

        render_unified(
            lineage_model,
            os.path.join(OUTPUT_FOLDER, f"{base}_unified")
        )

        render_drilldown(
            lineage_model,
            os.path.join(OUTPUT_FOLDER, f"{base}_dataset_drilldown")
        )

        print("Done.")
3️⃣ parser.py (Clean Stage Extraction)
import re

def split_into_stages(text):

    pattern = r"// --- \[(.*?)\] \[Lines.*?\] ---([\s\S]*?)(?=// --- \[|$)"
    matches = re.findall(pattern, text)

    stages = []

    for header, body in matches:
        name_match = re.search(r":\s*(.*)", header)
        if not name_match:
            continue

        stage_id = name_match.group(1).strip()

        stages.append({
            "stage_id": stage_id,
            "block": body.strip()
        })

    return stages
4️⃣ extractor.py (Deterministic Raw Extraction)
import re

def extract_raw_blocks(block):

    sql_match = re.search(r"SQL:(.*?)(Output:|StageType:|$)", block, re.DOTALL)
    sql_block = sql_match.group(1) if sql_match else ""

    transform_match = re.search(r"Transformations:(.*?)(Output:|$)", block, re.DOTALL)
    transform_block = transform_match.group(1) if transform_match else ""

    input_datasets = re.findall(
        r"Input:\s*←\s*dataset_\d+\s*\(([^)]+)\)", block
    )

    output_datasets = re.findall(
        r"Output:\s*→\s*dataset_\d+\s*\(([^)]+)\)", block
    )

    return {
        "sql": sql_block,
        "transform": transform_block,
        "inputs": input_datasets,
        "outputs": output_datasets
    }
5️⃣ llm_normalizer.py (Azure LLM Layer)
import os
import json
from dotenv import load_dotenv
from openai import AzureOpenAI

load_dotenv()

client = AzureOpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT")
)

DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT")


def normalize_stage(stage_id, raw):

    prompt = f"""
Extract lineage from SQL and transformation.

Return STRICT JSON:
{{
  "stage_id": "{stage_id}",
  "primary_tables": [],
  "joins": [
    {{
      "join_type": "",
      "table": "",
      "condition": ""
    }}
  ],
  "filters": [],
  "transformations": [
    {{
      "target": "",
      "logic": ""
    }}
  ]
}}

Rules:
- Extract top-level joins only
- Max 3 joins
- Copy exactly from text
- Ignore nested subqueries
"""

    response = client.chat.completions.create(
        model=DEPLOYMENT,
        temperature=0.0,
        messages=[
            {"role": "system", "content": "You extract structured lineage."},
            {"role": "user", "content": prompt + "\nSQL:\n" + raw["sql"] + "\nTRANSFORM:\n" + raw["transform"]}
        ]
    )

    content = response.choices[0].message.content.strip()

    try:
        return json.loads(content)
    except:
        return {
            "stage_id": stage_id,
            "primary_tables": [],
            "joins": [],
            "filters": [],
            "transformations": []
        }
6️⃣ validator.py
def validate_stage(normalized, raw):

    sql_text = raw["sql"]
    transform_text = raw["transform"]

    valid_joins = []

    for j in normalized.get("joins", []):
        if j["condition"] in sql_text:
            valid_joins.append(j)

    normalized["joins"] = valid_joins[:3]

    valid_transforms = []
    for t in normalized.get("transformations", []):
        if t["logic"] in transform_text:
            valid_transforms.append(t)

    normalized["transformations"] = valid_transforms[:6]

    return normalized
7️⃣ graph_builder.py
def build_lineage_model(stages):

    return stages

(Simple model — renderer consumes normalized stages)

8️⃣ renderer_unified.py
from graphviz import Digraph

def render_unified(stages, output_path):

    dot = Digraph("Unified", format="pdf")
    dot.attr(rankdir="LR")

    previous_dataset = None

    for stage in stages:

        label = stage["stage_id"] + "\n"

        for join in stage["joins"]:
            label += f"{join['join_type']} JOIN {join['table']}\n"
            label += f"  ON {join['condition']}\n"

        dot.node(stage["stage_id"], label,
                 shape="box", style="rounded,filled",
                 fillcolor="#FFE0B2")

        if previous_dataset:
            dot.edge(previous_dataset, stage["stage_id"])

        if stage["primary_tables"]:
            src_label = "\n".join(stage["primary_tables"])
            dot.node(stage["stage_id"] + "_src",
                     src_label,
                     shape="box",
                     style="filled",
                     fillcolor="#BBDEFB")
            dot.edge(stage["stage_id"] + "_src", stage["stage_id"])

        if stage.get("outputs"):
            ds = stage["outputs"][0]
            dot.node(ds, ds, shape="cylinder")
            dot.edge(stage["stage_id"], ds)
            previous_dataset = ds

    dot.render(output_path, cleanup=True)
9️⃣ renderer_drilldown.py
from graphviz import Digraph

def render_drilldown(stages, output_path):

    dot = Digraph("Drilldown", format="pdf")
    dot.attr(rankdir="TB")

    for stage in stages:

        header = f"DATASET: {stage.get('outputs', ['N/A'])[0]}"
        dot.node(header, header,
                 shape="box",
                 style="filled",
                 fillcolor="#90CAF9")

        stage_label = stage["stage_id"] + "\n"

        for join in stage["joins"]:
            stage_label += f"{join['join_type']} JOIN {join['table']}\n"
            stage_label += f"ON {join['condition']}\n"

        for t in stage["transformations"]:
            stage_label += f"{t['target']} ← {t['logic']}\n"

        dot.node(stage["stage_id"],
                 stage_label,
                 shape="box",
                 style="rounded,filled",
                 fillcolor="#FFE0B2")

        dot.edge(header, stage["stage_id"])

    dot.render(output_path, cleanup=True)


requirements.txt
graphviz
python-dotenv
openai
